In [1]:
# Install required packages
!pip install -q open_clip_torch tqdm huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00


In [2]:
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 39.2 MB/s eta 0:00:00:00:0100:01


In [3]:
import os
if not os.path.exists('/kaggle/working/daclip-uir'):
    !git clone https://github.com/Algolzw/daclip-uir.git /kaggle/working/daclip-uir

import sys
sys.path.insert(0, '/kaggle/working/daclip-uir/universal-image-restoration')
print('Path set. DA-CLIP repo ready.')

Cloning into '/kaggle/working/daclip-uir'...
remote: Enumerating objects: 532, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 532 (delta 93), reused 76 (delta 76), pack-reused 399 (from 1)
Receiving objects: 100% (532/532), 17.41 MiB | 21.20 MiB/s, done.
Resolving deltas: 100% (184/184), done.
Path set. DA-CLIP repo ready.


In [4]:
import os
import json
import numpy as np
import torch
import faiss
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import open_clip
import gc

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
print(f'FAISS version   : {faiss.__version__}')

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
FAISS version   : 1.14.1


In [5]:
# These MUST match the local inference pipeline exactly.
PATCH_SIZE   = 64          
STRIDE       = 32          
MODEL_NAME   = 'daclip_ViT-B-32'
FAISS_METRIC = 'cosine'    

DATASET_DIRS = [
    '/kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR',
    # '/kaggle/input/flickr2k/Flickr2K_HR/',   # to add Flickr2K later
    # '/kaggle/input/coco2017/train2017/',      # to add COCO later
]
CHECKPOINT_PATH = '/kaggle/working/daclip_ViT-B-32.pt'
OUTPUT_DIR      = '/kaggle/working/'

BATCH_SIZE        = 256    
CHECKPOINT_EVERY  = 50     
DEVICE            = 'cuda' if torch.cuda.is_available() else 'cpu'

PARTIAL_EMB_PATH  = os.path.join(OUTPUT_DIR, '_partial_embeddings.npy')
PARTIAL_MAP_PATH  = os.path.join(OUTPUT_DIR, '_partial_patch_map.json')
PROCESSED_PATH    = os.path.join(OUTPUT_DIR, '_processed_images.json')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config ready.')
print(f'  patch_size={PATCH_SIZE}, stride={STRIDE}, model={MODEL_NAME}')
print(f'  device={DEVICE}, batch_size={BATCH_SIZE}')

Config ready.
  patch_size=64, stride=32, model=daclip_ViT-B-32
  device=cuda, batch_size=256


In [6]:
from huggingface_hub import hf_hub_download

if not os.path.exists(CHECKPOINT_PATH):
    print('Downloading DA-CLIP weights from HuggingFace...')
    hf_hub_download(
        repo_id='weblzw/daclip-uir-ViT-B-32-irsde',
        filename='daclip_ViT-B-32.pt',
        local_dir='/kaggle/working/',
    )
    print('Download complete.')
else:
    print('Weights already present, skipping download.')

daclip_ViT-B-32.pt:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Download complete.


In [12]:
import torch

_original_load = torch.load

def patched_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_load(*args, **kwargs)

torch.load = patched_load

In [8]:
torch.serialization.add_safe_globals([np.core.multiarray.scalar])

/tmp/ipykernel_55/3561346164.py:1: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  torch.serialization.add_safe_globals([np.core.multiarray.scalar])


In [13]:
print(f'Loading {MODEL_NAME} from {CHECKPOINT_PATH} ...')

model, preprocess = open_clip.create_model_from_pretrained(
    MODEL_NAME,
    pretrained=CHECKPOINT_PATH,
)

model = model.to(DEVICE)
model.eval()

Loading daclip_ViT-B-32 from /kaggle/working/daclip_ViT-B-32.pt ...


DaCLIP(
  (clip): CLIP(
    (visual): VisionTransformer(
      (patchnorm_pre_ln): Identity()
      (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (patch_dropout): Identity()
      (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): ModuleList(
          (0-11): 12 x ResidualAttentionBlock(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
            )
            (ls_1): Identity()
            (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=768, out_features=3072, bias=True)
              (gelu): GELU(approximate='none')
              (c_proj): Linear(in_features=3072, out_features=768, bias=True)
            )
            (ls_2): Iden

In [14]:
dummy_pil = Image.fromarray(np.zeros((PATCH_SIZE, PATCH_SIZE, 3), dtype=np.uint8))
dummy_tensor = preprocess(dummy_pil).unsqueeze(0).to(DEVICE)

_, degra_feat = model.encode_image(dummy_tensor, control=True)

print(degra_feat.shape)

torch.Size([1, 512])


In [15]:
def extract_patches(pil_image: Image.Image, patch_size: int, stride: int):
    """
    Slide a window over the image and return (patches, coords).
    patches : list of PIL Images, each patch_size × patch_size
    coords  : list of (x, y) top-left corners
    """
    img = np.array(pil_image.convert('RGB'))
    H, W = img.shape[:2]
    patches, coords = [], []
    y = 0
    while y + patch_size <= H:
        x = 0
        while x + patch_size <= W:
            patch = img[y:y+patch_size, x:x+patch_size]
            patches.append(Image.fromarray(patch))
            coords.append((x, y))
            x += stride
        y += stride
    return patches, coords

@torch.no_grad()
def encode_patches(patches, model, preprocess, device, batch_size):
    """
    Encode a list of PIL patch images → numpy array of degra_features.
    Returns shape (N, EMBEDDING_DIM), L2-normalised.
    """
    all_feats = []
    for i in range(0, len(patches), batch_size):
        batch_pil  = patches[i:i+batch_size]
        batch_tensor = torch.stack([preprocess(p) for p in batch_pil]).to(device)
        with torch.cuda.amp.autocast():
            _, degra_features = model.encode_image(batch_tensor, control=True)
        # L2 normalise → cosine similarity via inner product
        degra_features = degra_features / degra_features.norm(dim=-1, keepdim=True)
        all_feats.append(degra_features.cpu().float().numpy())
    return np.concatenate(all_feats, axis=0)  # (N, D)

def load_checkpoint():
    """Load partial embeddings, patch_map and processed image list if available."""
    emb_list, patch_map, processed = [], {}, []

    if os.path.exists(PROCESSED_PATH):
        with open(PROCESSED_PATH) as f:
            processed = json.load(f)
        print(f'[Resume] Found checkpoint: {len(processed)} images already processed.')

    if os.path.exists(PARTIAL_EMB_PATH) and processed:
        emb_arr = np.load(PARTIAL_EMB_PATH)
        emb_list = [emb_arr]          # single chunk; we'll concatenate later
        print(f'[Resume] Loaded {emb_arr.shape[0]} partial embeddings.')

    if os.path.exists(PARTIAL_MAP_PATH) and processed:
        with open(PARTIAL_MAP_PATH) as f:
            patch_map = json.load(f)
        print(f'[Resume] Loaded {len(patch_map)} patch_map entries.')

    return emb_list, patch_map, processed


def save_checkpoint(emb_list, patch_map, processed):
    """Save partial state to disk."""
    combined = np.concatenate(emb_list, axis=0) if emb_list else np.empty((0, EMBEDDING_DIM), dtype=np.float32)
    np.save(PARTIAL_EMB_PATH, combined)
    with open(PARTIAL_MAP_PATH, 'w') as f:
        json.dump(patch_map, f)
    with open(PROCESSED_PATH, 'w') as f:
        json.dump(processed, f)


print('Utility functions defined.')

Utility functions defined.


In [16]:
VALID_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

image_paths = []
for d in DATASET_DIRS:
    dp = Path(d)
    if not dp.exists():
        print(f'[WARNING] Dataset directory not found: {d}')
        continue
    found = sorted([str(p) for p in dp.rglob('*') if p.suffix.lower() in VALID_EXTS])
    print(f'  {d}: {len(found)} images')
    image_paths.extend(found)

print(f'Total images to process: {len(image_paths)}')

  /kaggle/input/datasets/joe1995/div2k-dataset/DIV2K_train_HR: 800 images
Total images to process: 800


In [17]:

emb_chunks, patch_map, processed_set = load_checkpoint()
processed_set = set(processed_set)  # fast lookup
global_patch_idx = len(patch_map)   # next patch_map key

remaining = [p for p in image_paths if p not in processed_set]
print(f'Images remaining: {len(remaining)} / {len(image_paths)}')

images_since_ckpt = 0

for img_path in tqdm(remaining, desc='Processing images', unit='img'):
    img_name = Path(img_path).name
    try:
        pil_img = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f'[SKIP] Cannot open {img_path}: {e}')
        continue

    # Skip images too small for even one patch
    W, H = pil_img.size
    if H < PATCH_SIZE or W < PATCH_SIZE:
        print(f'[SKIP] Too small ({W}×{H}): {img_name}')
        processed_set.add(img_path)
        continue

    patches, coords = extract_patches(pil_img, PATCH_SIZE, STRIDE)
    if len(patches) == 0:
        continue

    embs = encode_patches(patches, model, preprocess, DEVICE, BATCH_SIZE)  # (N, D)

    # Accumulate
    emb_chunks.append(embs)
    for (x, y) in coords:
        patch_map[str(global_patch_idx)] = {'image': img_name, 'x': int(x), 'y': int(y)}
        global_patch_idx += 1

    processed_set.add(img_path)
    images_since_ckpt += 1

    # Periodic checkpoint
    if images_since_ckpt >= CHECKPOINT_EVERY:
        save_checkpoint(emb_chunks, patch_map, list(processed_set))
        print(f'  [Checkpoint] {global_patch_idx} patches saved so far.')
        images_since_ckpt = 0

    # Free GPU cache periodically
    if global_patch_idx % (CHECKPOINT_EVERY * 10) == 0:
        torch.cuda.empty_cache()
        gc.collect()

save_checkpoint(emb_chunks, patch_map, list(processed_set))
print(f'\nEncoding complete. Total patches: {global_patch_idx}')

Images remaining: 800 / 800


Processing images:   0%|          | 0/800 [00:00<?, ?img/s]

/tmp/ipykernel_55/53120797.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  [Checkpoint] 130386 patches saved so far.
  [Checkpoint] 257238 patches saved so far.
  [Checkpoint] 383842 patches saved so far.
  [Checkpoint] 511872 patches saved so far.
  [Checkpoint] 639964 patches saved so far.
  [Checkpoint] 766072 patches saved so far.
  [Checkpoint] 891746 patches saved so far.
  [Checkpoint] 1019032 patches saved so far.
  [Checkpoint] 1148178 patches saved so far.
  [Checkpoint] 1278378 patches saved so far.
  [Checkpoint] 1405540 patches saved so far.
  [Checkpoint] 1531772 patches saved so far.
  [Checkpoint] 1661166 patches saved so far.
  [Checkpoint] 1791986 patches saved so far.
  [Checkpoint] 1921814 patches saved so far.
  [Checkpoint] 2050526 patches saved so far.

Encoding complete. Total patches: 2050526


In [18]:
print('Running determinism validation...')

test_img_path = image_paths[0]
test_pil = Image.open(test_img_path).convert('RGB')
test_patches, _ = extract_patches(test_pil, PATCH_SIZE, STRIDE)
test_patch = test_patches[:1]  # single patch

emb1 = encode_patches(test_patch, model, preprocess, DEVICE, BATCH_SIZE)
emb2 = encode_patches(test_patch, model, preprocess, DEVICE, BATCH_SIZE)

max_diff = np.abs(emb1 - emb2).max()
assert max_diff < 1e-5, f'Embeddings are NOT deterministic! Max diff: {max_diff}'
print(f'  Determinism check PASSED (max diff = {max_diff:.2e})')

# Reconstruct final embedding array
all_embeddings = np.concatenate(emb_chunks, axis=0).astype(np.float32)
print(f'  Embedding array shape : {all_embeddings.shape}')  # (N, D)
print(f'  Embedding dim         : {all_embeddings.shape[1]}')
print(f'  Total patches indexed : {all_embeddings.shape[0]}')
assert all_embeddings.shape[0] == len(patch_map), \
    f'Mismatch: {all_embeddings.shape[0]} embeddings vs {len(patch_map)} patch_map entries'
print('  patch_map size matches embedding count. ✓')

Running determinism validation...


/tmp/ipykernel_55/53120797.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Determinism check PASSED (max diff = 0.00e+00)
  Embedding array shape : (2050526, 512)
  Embedding dim         : 512
  Total patches indexed : 2050526
  patch_map size matches embedding count. ✓


In [19]:
print('Building FAISS index...')

d = all_embeddings.shape[1]  # embedding dimension

# IndexFlatIP: exact inner-product search.
index = faiss.IndexFlatIP(d)

# Move to GPU for fast insertion (optional but fast)
if torch.cuda.is_available():
    res = faiss.StandardGpuResources()
    index_gpu = faiss.index_cpu_to_gpu(res, 0, index)
    index_gpu.add(all_embeddings)
    index = faiss.index_gpu_to_cpu(index_gpu)  # save as CPU index
    del index_gpu
    torch.cuda.empty_cache()
else:
    index.add(all_embeddings)

print(f'FAISS index built. ntotal = {index.ntotal}')
assert index.ntotal == all_embeddings.shape[0], 'FAISS ntotal mismatch!'
print('FAISS index validated. ✓')

Building FAISS index...
FAISS index built. ntotal = 2050526
FAISS index validated. ✓


In [21]:
index_path = os.path.join(OUTPUT_DIR, 'clean_patches.index')
faiss.write_index(index, index_path)
print(f'Saved: {index_path}')

emb_path = os.path.join(OUTPUT_DIR, 'embeddings.npy')
np.save(emb_path, all_embeddings)
print(f'Saved: {emb_path}')

map_path = os.path.join(OUTPUT_DIR, 'patch_map.json')
with open(map_path, 'w') as f:
    json.dump(patch_map, f, indent=2)
print(f'Saved: {map_path}')

config = {
    'patch_size'    : PATCH_SIZE,
    'stride'        : STRIDE,
    'model_name'    : MODEL_NAME,
    # 'embedding_dim' : int(EMBEDDING_DIM),
    'faiss_metric'  : FAISS_METRIC,
    'faiss_index_type': 'IndexFlatIP',
    'normalised'    : True,
    'encoding_call' : 'model.encode_image(image, control=True) -> degra_features',
    'num_patches'   : int(all_embeddings.shape[0]),
    'num_images'    : len(processed_set),
    'datasets'      : DATASET_DIRS,
}
cfg_path = os.path.join(OUTPUT_DIR, 'config.json')
with open(cfg_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'Saved: {cfg_path}')

# Clean up intermediate checkpoint files
for tmp in [PARTIAL_EMB_PATH, PARTIAL_MAP_PATH, PROCESSED_PATH]:
    if os.path.exists(tmp):
        os.remove(tmp)

print(f'  clean_patches.index : {os.path.getsize(index_path)/1e6:.1f} MB')
print(f'  embeddings.npy      : {os.path.getsize(emb_path)/1e6:.1f} MB')
print(f'  patch_map.json      : {os.path.getsize(map_path)/1e6:.2f} MB')
print(f'  config.json         : {os.path.getsize(cfg_path)/1e3:.1f} KB')
print(f'  Total patches       : {all_embeddings.shape[0]}')
print(f'  Embedding dim       : {all_embeddings.shape[1]}')

Saved: /kaggle/working/clean_patches.index
Saved: /kaggle/working/embeddings.npy
Saved: /kaggle/working/patch_map.json
Saved: /kaggle/working/config.json
  clean_patches.index : 4199.5 MB
  embeddings.npy      : 4199.5 MB
  patch_map.json      : 147.57 MB
  config.json         : 0.4 KB
  Total patches       : 2050526
  Embedding dim       : 512
